# 05. 주차장 추천 직접 해 보기 (TMAP 연동)

출발지와 목적지를 입력하면 도착 시점 기준으로 목적지 주변 공영주차장 상위 5곳을 추천한다.

| 단계 | 내용 | TMAP 호출 |
|---|---|---|
| ① 장소 검색 | 출발지, 목적지 텍스트 → 좌표 | POI 2건 |
| ② 자동차 경로 | 출발지 → 목적지 시간 → 주차장별 도착 시각 | 경로 1건 |
| ③ 후보 | 목적지 반경 1km (모자라면 2km), 가는 길의 환승주차장 | 없음 |
| ④ 예측 | 도착 시각의 예상 빈자리, 빈자리 3대 이상일 확률 | 없음 |
| ⑤ 상위 5곳 도보 경로 | 주차장 → 목적지 실제 도보 거리, 시간 | 경로 최대 5건 |

준비: `서울시 공영주차장 예측/.env` 파일에 `TMAP_APP_KEY=...` (이미 완료).
Free 요금제 경로 한도는 하루 1,000건이라 검색 1번에 최대 6건, 하루 약 160번 검색할 수 있다. 같은 검색은 10분 동안 다시 부르지 않는다.

## 1. 준비 (한 번만)

In [ ]:
%load_ext autoreload
%autoreload 2
# 위 두 줄: jariml 코드 파일이 바뀌면 커널 재시작 없이 자동으로 다시 불러온다
import os, sys
os.environ["LOKY_MAX_CPU_COUNT"] = "4"        # 윈도우 CPU 개수 확인 경고(wmic) 끄기
sys.path.insert(0, os.path.abspath(".."))
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from jariml import Predictor
from jariml.recommend import recommend, to_korean
from jariml.tmap import Tmap

LIVE = True                       # False면 TMAP을 부르지 않고 직선거리로 추정 (한도 절약, 장소는 "위도,경도"로 입력)
predictor = Predictor.load()
tmap = Tmap(live=LIVE)
print("운영 모델:", predictor.engine.name, "/ 학습 데이터 끝:", predictor.a["trained_until"])
print("TMAP 키:", "찾음" if tmap.key else "없음", "/ 오늘 호출 수:", Tmap.usage_today())

## 2. 연결 확인 (TMAP 장소 검색 1건)

오류가 나면 키나 상품 신청 문제다. 오류 메시지를 그대로 공유한다.

In [ ]:
tmap.place("서울시청")

## 3. 최신 주차 기록 받기

GitHub 수집 저장소에서 어제, 오늘 기록만 받는다 (약 5MB). 추천 전에 매번 다시 실행하면 가장 최신 기록으로 예측한다.

In [ ]:
recent = predictor.recent_from_github()
data_time = recent.t.max()
print(f"최신 기록: {data_time:%Y-%m-%d %H:%M} ({(pd.Timestamp.now() - data_time).total_seconds() / 60:.0f}분 전)")

## 4. 추천 받기

셀을 실행하면 입력칸이 뜬다. 출발지, 목적지는 장소 이름(예: 강남역, 코엑스) 또는 `37.4979,127.0276` 형태 좌표.
출발 시각은 비우면 지금이다. `0830`, `8:30`처럼 시각만 쓰면 오늘 그 시각(이미 지났으면 내일), `내일 0830`, `11/21 08:30`도 된다.

In [ ]:
ORIGIN = input("출발지: ").strip() or "강남역"
DEST = input("목적지: ").strip() or "서울시청"
DEPART = input("출발 시각 (비우면 지금, 예: 0830, 내일 0830, 11/21 08:30): ").strip() or None
STAY_MIN = int(input("주차 시간(분, 비우면 120): ").strip() or 120)

r = recommend(ORIGIN, DEST, predictor, recent, tmap, depart=DEPART, stay_min=STAY_MIN)
m = r["meta"]
print(f"출발지: {m['origin']['name']}  →  목적지: {m['dest']['name']}")
print(f"출발: {m['depart']:%m/%d %H:%M} / 운전 {m['car']['time_s'] / 60:.0f}분, {m['car']['dist_m'] / 1000:.1f}km ({m['car']['source']})")
print(f"예측 기준 기록: {m['data_time']:%H:%M} / 후보 {m['candidates']}곳 / TMAP 호출 {m['tmap_calls']} / 응답 {m['elapsed_s']}초")
if m["note"]:
    print("안내:", m["note"])
to_korean(r["table"])

## 5. 지도

빨간 별: 목적지, 검은 점: 출발지, 숫자: 추천 순위 (초록 높음, 주황 보통, 빨강 낮음, 파랑 테두리 = 환승주차장).

In [ ]:
import folium
t, o, d = r["table"], m["origin"], m["dest"]
fmap = folium.Map(location=[d["lat"], d["lon"]], zoom_start=15,
                  tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}", attr="Esri")
folium.Marker([d["lat"], d["lon"]], tooltip=f"목적지: {d['name']}", icon=folium.Icon(color="red", icon="star")).add_to(fmap)
folium.CircleMarker([o["lat"], o["lon"]], radius=6, color="black", fill=True, tooltip=f"출발지: {o['name']}").add_to(fmap)
COLOR = {"높음": "#2e7d32", "보통": "#ef6c00", "낮음": "#c62828"}
for _, x in t.iterrows():
    border = "3px solid #1565c0" if x.is_pnr else "2px solid white"
    html = (f'<div style="background:{COLOR[x.grade]};color:white;border:{border};border-radius:50%;width:26px;height:26px;'
            f'text-align:center;line-height:22px;font-weight:bold">{x["rank"]}</div>')
    folium.Marker([x.lat, x.lon], icon=folium.DivIcon(html=html, icon_size=(26, 26), icon_anchor=(13, 13)),
                  tooltip=f"{x['rank']}. {x['name']} / 확률 {x.prob}% / 빈자리 {x.pred_free:.0f}대 / {x.last_leg} {x.last_min}분").add_to(fmap)
fmap.fit_bounds([[min(t.lat.min(), d["lat"]), min(t.lon.min(), d["lon"])], [max(t.lat.max(), d["lat"]), max(t.lon.max(), d["lon"])]])
fmap

## 6. 오늘 TMAP 사용량

경로 한도 1,000건 중 몇 건 썼는지 본다.

In [ ]:
u = Tmap.usage_today()
print(f"장소 검색 {u['poi']:,} / 20,000건,  경로 {u['route']:,} / 1,000건 (남은 검색 약 {(1000 - u['route']) // 6}번)")